---
**Author:** Leonardo Gabriel Mourao Thiel  
**Project:** Master Thesis – System Inertia in the Energy System of the Future:
Model-Based Cost Optimization to Secure Inertia Requirements

**Notebook:**  System Inertia Analysis for the European Power System (2024)

**Date:** 2026  
---

# System Inertia Analysis for the European Power System (2024)

This notebook presents a data-driven analysis of system inertia
in the European power system for the year 2024.

The study is based on harmonized generation and cross-border flow data
for multiple countries and aims to quantify temporal and spatial
variations in system inertia.

---

## Background and Motivation

System inertia is a fundamental property of power systems, reflecting
their ability to resist frequency deviations following disturbances.

With the increasing penetration of renewable energy sources,
which typically do not provide inherent rotational inertia,
the overall inertia of the system is expected to decrease.
This raises concerns regarding frequency stability and system resilience.

Understanding how inertia evolves over time and across regions
is therefore essential for future system planning and operation.

---

## Objective

The objective of this analysis is to:

- quantify system inertia \( H_{sys}(t) \) for each country  
- identify temporal patterns and critical low-inertia periods  
- analyze spatial differences across the European system  
- establish a consistent data foundation for further stability studies  

---

## Methodological Approach

The analysis consists of the following main steps:

1. Harmonization of generation time series  
   - Integration of datasets with different temporal resolutions  
   - Conversion to a unified hourly timeline  

2. Reconstruction of system load  
   - Combination of generation and net imports  

3. Parameterization of inertia  
   - Assignment of technology-specific inertia constants  
   - Inclusion of loading factors  

4. Computation of system inertia  
   - Aggregation of contributions across technologies  

5. Export and evaluation of results  

---

## Key Assumptions

- Generation data represents power values (MW)  
- Aggregation from 15-minute to hourly resolution uses mean values  
- System load is reconstructed as:
  
  > Load = Generation + Net Imports  

- Only generation technologies contribute to inertia  
- A fixed timezone (CET, without daylight saving time) is used  
  to ensure temporal consistency  

---

## Scope

- Year: 2024 (full calendar year, including leap day)  
- Temporal resolution: hourly  
- Spatial scope: multi-country European system  
- Technologies: aggregated by production type  

---

## Output

The main output of this notebook is a time series dataset containing:

- hourly system inertia \( H_{sys}(t) \) per country  

This dataset is exported for further analysis, visualization,
and integration into subsequent studies.

---

## Reproducibility

All data processing steps are implemented within this notebook,
ensuring transparency and reproducibility of the analysis workflow.
External input data and parameters are explicitly defined and loaded.

---

## Notes

This notebook focuses on data processing and inertia computation.
Interpretation of results and visualization are performed in
separate analysis notebooks.

## 1. Environment Setup and Data Configuration

This section initializes the analysis environment and defines the
core configuration parameters for the system inertia analysis.

The required Python packages are installed and imported, and the
set of countries as well as the data source directory are specified.

The selected countries represent the interconnected European power system,
forming the basis for the subsequent inertia analysis.

In [1]:
# Install required Python packages from requirements file
# (ensures reproducibility of the analysis environment)
import sys

!{sys.executable} -m pip install -r requirements.txt

import pandas as pd
import os

# ---------------------------------------------------------
# Model scope: countries included in the analysis
# ---------------------------------------------------------

# List of countries considered in the European system
countryList = [
    "AT","BA","BE","BG","CH","CZ","DE","DK","ES","FR","GR"
]

# Extend list with additional countries
countryList += [
    "HR","HU","IT","LU","MK","ME","NL","PL","PT","RO","RS","SI","SK"
]


# ---------------------------------------------------------
# Data source configuration
# ---------------------------------------------------------

# Path to generation data used for inertia analysis
path = "..\\Data\\2024"

ERROR: Could not find a version that satisfies the requirement os (from versions: none)

[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: C:\Users\Leo\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for os


## 2. Harmonization of Generation Time Series (2024)

This section harmonizes national generation time series with differing
temporal resolutions into a consistent hourly dataset for the year 2024.

The input data varies across countries, with some datasets provided at
15-minute resolution and others at hourly resolution. To enable a
consistent system-wide analysis of inertia, all time series are converted
to a common hourly resolution.

---

### Methodological Approach

The harmonization process consists of the following steps:

1. Loading country-specific generation datasets  
2. Detecting temporal resolution based on dataset length  
3. Mapping values onto a fixed reference timeline  
4. Aggregating 15-minute data to hourly resolution  

---

### Key Assumptions

- The data represents **power values (MW)**  
- Aggregation from 15-minute to hourly resolution is performed using
  **arithmetic mean values**  
- A fixed timezone (CET, UTC+1 without daylight saving time) is used
  to avoid inconsistencies caused by DST transitions  

---

### Rationale

Using a unified hourly time base ensures comparability across countries
and avoids temporal misalignment, which is critical for subsequent
system-level analyses such as inertia evaluation.

In [2]:
import os
import sys
import pandas as pd
import pytz

# ============================================================
# Harmonization of generation time series to hourly resolution
# ============================================================


# ------------------------------------------------------------
# Define fixed CET timezone (UTC+1, no daylight saving time)
# Avoids ambiguities due to DST transitions
# ------------------------------------------------------------
cet_fixed = pytz.FixedOffset(60)


# ------------------------------------------------------------
# Reference timelines for 2024 (leap year)
# ------------------------------------------------------------

# 15-minute resolution: 366 days × 96 intervals
timeline_15m = pd.date_range(
    start="2024-01-01 00:00:00",
    end="2024-12-31 23:45:00",
    freq="15min",
    tz=cet_fixed
)

# Hourly resolution: 366 days × 24 hours
timeline_1h = pd.date_range(
    start="2024-01-01 00:00:00",
    end="2024-12-31 23:00:00",
    freq="1h",
    tz=cet_fixed
)


# ------------------------------------------------------------
# Storage structures
# ------------------------------------------------------------

# Final harmonized hourly data per country
country_hourly = {}



# ============================================================
# Processing loop over all countries
# ============================================================

for country in countryList:

    filepath = os.path.join(path, f"{country}.xlsx")

    # Handle missing files
    if not os.path.exists(filepath):
        print(f"File for {country} does not exist")
        continue

    # --------------------------------------------------------
    # Load dataset
    # --------------------------------------------------------
    df = pd.read_excel(filepath, header=0, skiprows=5, engine="openpyxl")

    # Clean column names (remove units and whitespace)
    df.columns = [str(c).replace(" (MW)", "").strip() for c in df.columns]


    # --------------------------------------------------------
    # Flatten multi-level column headers
    # --------------------------------------------------------
    columns = []
    for i, col in enumerate(df.columns):
        if isinstance(col, tuple):
            if i == 0:
                columns.append("MTU")
            else:
                columns.append(col[1] if col[1] != "" else col[0])
        else:
            columns.append(col)

    df.columns = columns


    # --------------------------------------------------------
    # Data cleaning
    # --------------------------------------------------------

    # Replace missing entries ("n/e") with zero
    df = df.replace("n/e", 0)

    # Identify numeric columns
    num_cols = [c for c in df.columns if c not in ["MTU", "Start"]]

    # Convert to numeric (coerce invalid values to NaN)
    for col in num_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    values_only = df[num_cols].reset_index(drop=True)


    # ========================================================
    # Detect temporal resolution
    # ========================================================

    n = len(values_only)

    # Heuristic threshold to distinguish 15-min vs hourly data
    if n > 20000:
        base_timeline = timeline_15m
        base_freq = "15min"
    else:
        base_timeline = timeline_1h
        base_freq = "1h"


    # --------------------------------------------------------
    # Map data onto fixed reference timeline
    # --------------------------------------------------------

    df_fixed = pd.DataFrame(
        index=base_timeline,
        columns=num_cols,
        dtype=float
    )

    # Align data by position (not timestamp)
    df_fixed.iloc[:min(n, len(df_fixed))] = values_only.iloc[:min(n, len(df_fixed))].values

    # Fill missing values with zero
    df_fixed = df_fixed.fillna(0)


    # ========================================================
    # Convert to hourly resolution
    # ========================================================

    if base_freq == "15min":
        # Aggregate 4×15min → 1h using mean (valid for MW values)
        hourly_cet = df_fixed.resample("1h").mean()
    else:
        hourly_cet = df_fixed.copy()


    # Store result
    country_hourly[country] = hourly_cet



# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------
print("Processing completed.")


Processing completed.


### Net Import Calculation per Country

This function computes the net electricity imports for a given country
based on cross-border flow data.

The dataset contains directional flow information between countries,
encoded in column names using the format:

    "CountryA -> CountryB"

Net imports are calculated as:

- total imports (flows into the country)  
- minus total exports (flows out of the country)  

---

### Methodological Notes

- Flow directions are inferred from column names  
- Missing or non-numeric values are handled via coercion  
- The time index is extracted and standardized from the MTU column  

---

### Definition

Net imports are defined as:

> Net Imports = Imports − Exports

Positive values indicate net importing behavior, while negative values
indicate net exporting.

In [3]:
def compute_net_imports(df: pd.DataFrame, country_code: str) -> pd.Series:
    """
    Compute net electricity imports for a given country.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing cross-border flow data. Column names are expected
        to follow the format "CountryA -> CountryB".
    country_code : str
        ISO country code (e.g., "DE", "FR") for which net imports are calculated.

    Returns
    -------
    pandas.Series
        Time series of net imports (imports - exports).
        Positive values indicate net imports, negative values net exports.
    """

    # Create a copy to avoid modifying the original DataFrame
    df = df.copy()

    # Clean column names (remove whitespace)
    df.columns = [str(c).strip() for c in df.columns]

    # -----------------------------
    # Time index extraction
    # -----------------------------

    # Extract start time from MTU column and convert to datetime
    df["time"] = pd.to_datetime(
        df["MTU"]
        .astype(str)
        .str.split(" - ").str[0]          # take start of interval
        .str.replace(r"\s*\(.*\)", "", regex=True)  # remove timezone text
        .str.strip(),
        dayfirst=True,
        errors="coerce"
    )

    # Set time as index
    df = df.set_index("time")

    # -----------------------------
    # Identify relevant flow columns
    # -----------------------------

    # Target country pattern (e.g., "(DE)")
    target = f"({country_code})"

    imports = []
    exports = []

    # Ensure numeric values in flow columns
    for col in df.columns:
        if "->" in col:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Classify flows as imports or exports
    for col in df.columns:

        if "->" not in col:
            continue

        try:
            from_c, to_c = col.split("->")
            from_c = from_c.strip()
            to_c = to_c.strip()
        except ValueError:
            # Skip malformed column names
            continue

        # Flow into country → import
        if target in to_c:
            imports.append(col)

        # Flow out of country → export
        elif target in from_c:
            exports.append(col)

    # -----------------------------
    # Compute net imports
    # -----------------------------

    # Sum imports and exports across all connections
    net_imports = df[imports].sum(axis=1) - df[exports].sum(axis=1)

    return net_imports

## 3. Integration of Net Imports and Load Estimation

In this section, national generation time series are combined with
cross-border electricity flows to reconstruct total system load.

For each country, net imports are calculated and added to total
generation in order to estimate the effective load:

> Total Load = Generation + Net Imports

---

### Data Processing Steps

For each country:

1. Load cross-border flow data  
2. Compute net imports using directional flow information  
3. Align time indices between generation and import data  
4. Merge datasets into a unified time series  
5. Compute total load  

---

### Data Validation

To identify potentially invalid data points, a simple heuristic is applied:

- load values below 20% of the mean are flagged  
- negative load values are considered invalid  

These flags help detect inconsistencies in the input data.

In [4]:
# ============================================================
# Merge generation data with net imports and compute load
# ============================================================

for country in countryList:

    # Skip countries without generation data
    if country not in country_hourly:
        print(f" {country} missing")
        continue

    # Copy generation data
    df = country_hourly[country].copy()

    # --------------------------------------------------------
    # Load import data
    # --------------------------------------------------------
    import_path = os.path.join(path, "Imports", f"{country}.xlsx")

    df_imports = pd.read_excel(
        import_path,
        header=0,
        skiprows=5,
        engine="openpyxl"
    )

    # Compute net imports (imports - exports)
    imports = compute_net_imports(df_imports, country)


    # --------------------------------------------------------
    # Time alignment
    # --------------------------------------------------------

    # Ensure datetime indices
    df.index = pd.to_datetime(df.index)
    imports.index = pd.to_datetime(imports.index)

    # Remove timezone information to ensure compatibility
    df.index = df.index.tz_localize(None)
    imports.index = imports.index.tz_localize(None)

    # Sort indices (important for alignment)
    df = df.sort_index()
    imports = imports.sort_index()

    # Remove duplicate timestamps in imports (average duplicates)
    imports = imports.groupby(imports.index).mean()

    # Align imports to generation timeline
    imports = imports.reindex(df.index)


    # --------------------------------------------------------
    # Load calculation
    # --------------------------------------------------------

    # Total generation (sum over all generation types)
    df["Total_Generation"] = df.sum(axis=1)

    # Add net imports
    df["Net_Imports"] = imports

    # Compute total load
    df["Total_Load"] = df["Total_Generation"] + df["Net_Imports"]


    # --------------------------------------------------------
    # Data validation
    # --------------------------------------------------------

    # Compute average load for threshold definition
    mean_load = df["Total_Load"].mean()

    # Flag unrealistic values:
    # - negative load
    # - very low load (<20% of mean)
    df["Invalid"] = (
        (df["Total_Load"] < mean_load * 0.2) |
        (df["Total_Load"] < 0)
    )


    # --------------------------------------------------------
    # Store updated dataset
    # --------------------------------------------------------
    country_hourly[country] = df



## 4. Inertia Parameters

This section defines the inertia-related parameters used in the analysis.

For each generation technology, two key parameters are specified:

- **Mean inertia constant (H [s])**  
- **Loading factor [-]**, representing the fraction of installed capacity
  that is effectively contributing to inertia  

---

### Methodological Role

These parameters are used to estimate the contribution of different
generation types to total system inertia.

The system inertia is later computed as an aggregation of individual
generation contributions, weighted by their respective inertia constants
and operational levels.

---

### Data Source and Processing

The parameters are imported from an external dataset and preprocessed
to ensure numerical consistency.

In [5]:
# ---------------------------------------------------------
# Load inertia parameter data
# ---------------------------------------------------------

inertia = pd.read_excel("..\\Data\\inputs\\inertia.xlsx")


# ---------------------------------------------------------
# Data cleaning: convert decimal format
# ---------------------------------------------------------

# Replace comma decimal separators (e.g., "5,9" → "5.9")
# and convert to float
inertia["Mean H [s]"] = (
    inertia["Mean H [s]"]
    .astype(str)
    .str.replace(",", ".")
    .astype(float)
)

inertia["Loading factor [-]"] = (
    inertia["Loading factor [-]"]
    .astype(str)
    .str.replace(",", ".")
    .astype(float)
)


# ---------------------------------------------------------
# Create lookup dictionaries
# ---------------------------------------------------------

# Map generation type → inertia constant
H_dict = dict(zip(inertia["Production Type"], inertia["Mean H [s]"]))

# Map generation type → loading factor
LF_dict = dict(zip(inertia["Production Type"], inertia["Loading factor [-]"]))



## 5. Computation of System Inertia (H_sys)

This section defines the computation of system inertia at each time step.

System inertia is calculated as a weighted aggregation of the inertia
constants of individual generation technologies, based on their
relative contribution to total system generation.

---

### Definition

The system inertia \( H_{sys}(t) \) is defined as:

- a weighted average of technology-specific inertia constants  
- scaled by their respective generation shares  

---

### Methodological Assumptions

- Only active generation contributes to system inertia  
- Contributions are weighted by relative generation output  
- A loading factor is applied to account for partial utilization  
- Invalid or inconsistent data points are excluded  

---

### Interpretation

- Higher \( H_{sys} \) indicates greater resistance to frequency deviations  
- Lower values indicate potential system vulnerability  

In [6]:
import numpy as np

def compute_H_sys(hour_row: pd.Series) -> float:
    """
    Compute system inertia H_sys(t) for a single timestep.

    Parameters
    ----------
    hour_row : pandas.Series
        One row of the hourly generation DataFrame.
        Contains generation per technology and auxiliary columns.

    Returns
    -------
    float
        System inertia H_sys(t). Returns np.nan for invalid cases.
    """

    # --------------------------------------------------------
    # 1) Determine total system generation S_tot(t)
    # --------------------------------------------------------

    # Prefer precomputed total if available
    if "Total_Generation" in hour_row:
        S_tot = hour_row["Total_Generation"]
    else:
        # Sum only relevant generation technologies
        tech_cols = [c for c in hour_row.index if c in H_dict and c in LF_dict]
        S_tot = hour_row[tech_cols].sum()

    # --------------------------------------------------------
    # 2) Data validation
    # --------------------------------------------------------

    # Skip invalid flagged rows
    if "Invalid" in hour_row and hour_row["Invalid"]:
        return np.nan

    # Handle missing or zero generation
    if pd.isna(S_tot) or S_tot <= 0:
        return np.nan


    # --------------------------------------------------------
    # 3) Compute weighted system inertia
    # --------------------------------------------------------

    H_sys = 0.0

    for tech, S_i in hour_row.items():

        # Skip non-generation columns
        if tech not in H_dict or tech not in LF_dict:
            continue

        # Skip invalid or zero generation
        if pd.isna(S_i) or S_i <= 0:
            continue

        H_i = H_dict[tech]
        LF_i = LF_dict[tech]

        # Skip invalid parameters
        if pd.isna(H_i) or pd.isna(LF_i) or LF_i == 0:
            continue

        # Weight factor (generation share)
        w_i = S_i / S_tot

        # Contribution to system inertia
        H_sys += (H_i * w_i) / LF_i

    return float(H_sys)

## 6. Computation of System Inertia Time Series

In this section, the system inertia \(H_{sys}(t)\) is computed for all
countries and time steps.

For each country:

- the hourly generation data is processed  
- the inertia function is applied to each time step  
- results are aggregated into a multi-country dataset  

---

### Output Structure

The resulting dataset contains:

- rows: hourly timestamps  
- columns: countries  
- values: computed system inertia \(H_{sys}(t)\)  

---

### Postprocessing

The results are exported to an Excel file for further analysis and
visualization. Time indices are converted to timezone-naive format
to ensure compatibility with external tools.

In [7]:
import os
import pandas as pd

# --------------------------------------------------------
# Initialize result container
# --------------------------------------------------------

# Use index from first available country as reference timeline
H_sys_all = pd.DataFrame(
    index=country_hourly[next(iter(country_hourly))].index
)


# --------------------------------------------------------
# Loop over countries
# --------------------------------------------------------

for country in countryList:

    # Skip missing countries
    if country not in country_hourly:
        print(f"{country} missing.")
        continue

    df = country_hourly[country]


    # --------------------------------------------------------
    # Compute H_sys for each hour
    # --------------------------------------------------------

    # Apply function row-wise (cleaner than manual loop)
    H_country = df.apply(compute_H_sys, axis=1)

    # Store results
    H_sys_all[country] = H_country



# --------------------------------------------------------
# Prepare export (Excel compatibility)
# --------------------------------------------------------

# Reset index (convert DatetimeIndex → column)
H_sys_all = H_sys_all.reset_index()

# Rename column
H_sys_all.rename(columns={'index': 'Datetime'}, inplace=True)

# Ensure timezone-naive datetime
H_sys_all['Datetime'] = pd.to_datetime(H_sys_all['Datetime']).dt.tz_localize(None)


# --------------------------------------------------------
# Save results
# --------------------------------------------------------

output_folder = os.path.join("..", "Results","2024_inertia")
os.makedirs(output_folder, exist_ok=True)

output_file = os.path.join(output_folder, "H_sys_hourly_all_countries.xlsx")

H_sys_all.to_excel(output_file, index=False)



## 7. Export of Harmonized Country Data

This section exports the processed and harmonized country-level datasets
to a multi-sheet Excel file.

Each country is stored in a separate worksheet, containing:

- hourly generation data  
- net imports  
- reconstructed total load  
- data validity flags  

---

### Purpose

The exported dataset serves as:

- a basis for further analysis and visualization  
- a reproducible data source for validation  
- supplementary material for the thesis  

---

### Data Handling

To ensure compatibility with Excel:

- timezone information is removed from datetime indices  
- indices are converted to explicit columns  

In [8]:
import os
import pandas as pd

# ---------------------------------------------------------
# Output file definition
# ---------------------------------------------------------

output_file = os.path.join(path, "country_hourly_all.xlsx")


# ---------------------------------------------------------
# Write multi-sheet Excel file
# ---------------------------------------------------------

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    for country, df in country_hourly.items():

        df_to_save = df.copy()

        # ----------------------------------------------------
        # Ensure timezone-naive datetime index (Excel-safe)
        # ----------------------------------------------------
        if getattr(df_to_save.index, "tz", None) is not None:
            df_to_save.index = df_to_save.index.tz_localize(None)

        # ----------------------------------------------------
        # Convert index to column
        # ----------------------------------------------------
        df_to_save = df_to_save.reset_index()
        df_to_save.rename(columns={"index": "Datetime"}, inplace=True)

        # ----------------------------------------------------
        # Excel sheet name constraints (max 31 chars)
        # ----------------------------------------------------
        sheet_name = str(country)[:31]

        # ----------------------------------------------------
        # Write to Excel
        # ----------------------------------------------------
        df_to_save.to_excel(writer, sheet_name=sheet_name, index=False)

